# Generative AI

## Gen AI Topics:
1. Gen AI Introduction
2. Langchain and LangGraph
3. Prompt Engineering
4. LLM Fine-tunning Methods
5. RAG (Retrival Augumented Generation)
6. Agentic AI
7. Evalution Metrics
8. Hallucination Control
9. MCP (Model Context Protocol)

### Simple Chatbot

In [48]:
import requests

def llm_chatbot(user_query):
    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "phi3:mini",
        "messages": [
            {
                "role": "system",
                "content": """You are helpfull support ticket Assistant.
                Generate reponse carefully:
                """
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        "stream": False,
        "options": {
            "num_predict": 150
        }
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()

    return response.json()["message"]["content"]

In [49]:
print(llm_chatbot("I have one ticket related my mobile issue"))

Hello! I'm here to assist you with your mobile issues. Could you please provide more details about the problem? Knowing specifics like error messages, app behaviors or hardware symptoms will help me understand and better guide you through a solution. Meanwhile, could we start by checking if all cables are firmly connected and if not, try reconnecting them to your device while I prepare some standard troubleshooting steps for us?


In [45]:
print(llm_chatbot("Write python Recursion Code"))

Certainly! Below is a simple Python function that uses recursion to calculate the factorial of a number `n`. Please note that this example assumes you're familiar with basic concepts in programming like functions, conditional statements (if-else), and returning values from functions. The comments provide additional explanation at each step for better understanding:

```python
def recursive_factorial(n):
    # Base case: factorial of 1 is itself
    if n == 1:  
        return 1
    
    # Recursive case: n! = n * (n-1)!
    else:
        result = n * recursive_factorial(n - 1)
        
       


### Langchain

In [50]:
pip install langchain


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
pip install langchain.llm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached jiter-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 3.7 MB/s  0:00:00m 4.2 MB/s eta 0:00:01
Using cached jiter-0.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (360 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 4.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.4 MB/s  0:00:00m 4.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 5.1 MB/s  0:00:02a 0:00:0136m0:00:01
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Created wheel for transformers_stream_generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12526 sha256=2f42368389ad02a6d57e11ecb942c3ae870f9881f8ca910bfae46733f7

## RAG

### RAG Architecture Component:
1. Document Loading
2. Text Spliting (Chunking)
3. Embedding (Convert chunking in vector value)
4. Vector Database (Store embedding in vector database)
5. Query Embedding 
6. Semantic search   (Cosine similarity)
7. Retive Top k Results
8. Show response to user (Prompt engineering, user question, Top-K results, LLM)

In [1]:
from langchain_community.document_loaders import TextLoader

#1. Document Loading

loader=TextLoader("data.txt")
docs=loader.load()
print(docs)

/home/hp/Documents/AIML/AIML_Nov/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'data.txt'}, page_content='\n\n# Salesforce Detailed Cheat Sheet\n\n## 1. What is Salesforce?\n\n**Salesforce** is a **cloud-based Customer Relationship Management (CRM) platform** used to manage:\n\n* Sales\n* Customer service\n* Marketing\n* Analytics\n* Application development\n\nIt runs entirely on the cloud so users access it through a browser without installing software.\n\n**Key Benefits**\n\n* Cloud-based (SaaS)\n* Highly customizable\n* Scalable\n* Large ecosystem (AppExchange)\n\n---\n\n# 2. Salesforce Architecture\n\n### Multi-Tenant Architecture\n\nSalesforce uses **multi-tenancy**, meaning multiple customers share the same infrastructure.\n\nExample:\n\n* Like living in an apartment building\n* Everyone shares resources but data is secure.\n\n### Metadata Driven Architecture\n\nSalesforce apps are built using **metadata instead of code**.\n\nExamples of metadata:\n\n* Objects\n* Fields\n* Page Layouts\n* Workflows\n* Validation Rules\n\nAdvant

In [26]:
from langchain_community.document_loaders import TextLoader

#1. Document Loading

loader=TextLoader("data.txt")
docs=loader.load()
print(docs)

from langchain_text_splitters import RecursiveCharacterTextSplitter

#2. Text Spliting (Chunking)
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap= 30
)

rag_splits=rag_splitter.split_documents(docs)
print(len(rag_splits))

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

#3. Embedding (Convert chunking in vector value)
embedding= OllamaEmbeddings(model="nomic-embed-text")

#4. Vector Database (Store embedding in vector database)

vectorestore=Chroma.from_documents(
    rag_splits,
    embedding,
    persist_directory="./chroma_db"
)


[Document(metadata={'source': 'data.txt'}, page_content='\n\n# Salesforce Detailed Cheat Sheet\n\n## 1. What is Salesforce?\n\n**Salesforce** is a **cloud-based Customer Relationship Management (CRM) platform** used to manage:\n\n* Sales\n* Customer service\n* Marketing\n* Analytics\n* Application development\n\nIt runs entirely on the cloud so users access it through a browser without installing software.\n\n**Key Benefits**\n\n* Cloud-based (SaaS)\n* Highly customizable\n* Scalable\n* Large ecosystem (AppExchange)\n\n---\n\n# 2. Salesforce Architecture\n\n### Multi-Tenant Architecture\n\nSalesforce uses **multi-tenancy**, meaning multiple customers share the same infrastructure.\n\nExample:\n\n* Like living in an apartment building\n* Everyone shares resources but data is secure.\n\n### Metadata Driven Architecture\n\nSalesforce apps are built using **metadata instead of code**.\n\nExamples of metadata:\n\n* Objects\n* Fields\n* Page Layouts\n* Workflows\n* Validation Rules\n\nAdvant

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#2. Text Spliting (Chunking)
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap= 50
)

rag_splits=rag_splitter.split_documents(docs)
print(len(rag_splits))

16


In [3]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

#3. Embedding (Convert chunking in vector value)
embedding= OllamaEmbeddings(model="nomic-embed-text")

#4. Vector Database (Store embedding in vector database)

vectorestore=Chroma.from_documents(
    rag_splits,
    embedding,
    persist_directory="./chroma_db"
)


/tmp/ipykernel_213130/2270320855.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding= OllamaEmbeddings(model="nomic-embed-text")


In [4]:
retriver=vectorestore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)

print(retriver)

tags=['Chroma', 'OllamaEmbeddings'] vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ae70400d3a0> search_type='mmr' search_kwargs={'k': 3, 'fetch_k': 10}


In [5]:
query="What is Salesforce and what are its key benefits?"

retrive_results=retriver.invoke(query)

print(retrive_results)

[Document(metadata={'source': 'data.txt'}, page_content='# Salesforce Detailed Cheat Sheet\n\n## 1. What is Salesforce?\n\n**Salesforce** is a **cloud-based Customer Relationship Management (CRM) platform** used to manage:\n\n* Sales\n* Customer service\n* Marketing\n* Analytics\n* Application development\n\nIt runs entirely on the cloud so users access it through a browser without installing software.\n\n**Key Benefits**\n\n* Cloud-based (SaaS)\n* Highly customizable\n* Scalable\n* Large ecosystem (AppExchange)\n\n---\n\n# 2. Salesforce Architecture\n\n### Multi-Tenant Architecture'), Document(metadata={'source': 'data.txt'}, page_content='| Field Type    | Example                                 |\n| ------------- | --------------------------------------- |\n| Text          | Name                                    |\n| Number        | Quantity                                |\n| Email         | [user@email.com](mailto:user@email.com) |\n| Phone         | Phone number                

In [40]:
rag_query=llm_chatbot("what is mmr in search_type, could you please this retriver arguement")

In [41]:
print(rag_query)

Main Points:  
- MMR stands for Minimax Regret when discussing ranking lists or recommendations within a retrieval system. It's not typically used as an argument directly but rather as the objective to optimize in such systems, aiming at minimizing regret (or loss) relative to an ideal best ordering based on user preferences and item relevance scores provided by implicit feedback mechanisms like clicks and dwell times.
- MMR is closely related to concepts of Expected Utility Maximization or even Best Arm Identification but applied in a ranking context where the choices are not independent actions, as with these methods typically used for bandit problems. Instead, it's about sequentially deciding which item should be


In [6]:
import requests

def llm_chatbot(user_query):
    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "phi3:mini",
        "messages": [
            {
                "role": "system",
                "content": """You are helpfull support Assistant.
                Generate reponse point wise like below:
                Main Points:
                    1. Sub Points
                    2. Sub Points
                    3. Sub Points
                """
            },
            {
                "role": "user",
                "content": user_query
            }
        ],
        "stream": False,
        "options": {
            "num_predict": 150
        }
    }

    response = requests.post(url, json=payload)
    response.raise_for_status()

    return response.json()["message"]["content"]

In [7]:
context= "\n".join([doc.page_content for doc in retrive_results])
print(context)

# Salesforce Detailed Cheat Sheet

## 1. What is Salesforce?

**Salesforce** is a **cloud-based Customer Relationship Management (CRM) platform** used to manage:

* Sales
* Customer service
* Marketing
* Analytics
* Application development

It runs entirely on the cloud so users access it through a browser without installing software.

**Key Benefits**

* Cloud-based (SaaS)
* Highly customizable
* Scalable
* Large ecosystem (AppExchange)

---

# 2. Salesforce Architecture

### Multi-Tenant Architecture
| Field Type    | Example                                 |
| ------------- | --------------------------------------- |
| Text          | Name                                    |
| Number        | Quantity                                |
| Email         | [user@email.com](mailto:user@email.com) |
| Phone         | Phone number                            |
| Picklist      | Status                                  |
| Checkbox      | True/False                              |
| Date      

In [8]:
prompt=f"""
Answer the question using below context.
context: {context}
Question: {query}
"""

rag_results=llm_chatbot(prompt)

print(rag_results)

Main Points:

1. Definition of Salesforce
   - Sub Points:
     * A cloud-based Customer Relationship Management (CRM) platform that helps in managing various business aspects like sales, customer service, marketing, analytics and application development. It does not require any software installation as users can access it through a browser on the web from anywhere with an internet connection.
2. Key Benefits of Salesforce:
   - Sub Points:
     * Cloud-based (SaaS) to enable easy remote work without worrying about hardware and infrastructure maintenance or upgrades, leading to cost savings.
     * Highly customizable software that can be tail
